In [1]:
# [Cell 1] Install dependensi terbaru
!pip install --upgrade ultralytics roboflow onnx onnxruntime-gpu --quiet

import os
import torch
import shutil
from ultralytics import YOLO

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 7.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
CUDA Available: True
Device Count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [2]:
# [Cell 2] Unduh Dataset dari Roboflow
from roboflow import Roboflow

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("roboflow_spoonbill_gilang_API")

# Ganti dengan API Key Anda jika diperlukan
ROBOFLOW_API_KEY = secret_value_0  # Masukkan API Key Anda

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("blackfaced-spoonbill").project("my-first-project-cn7qx")
version = project.version(4)

# Format 'yolov11' atau 'yolov8' untuk task segmentasi poligon
dataset = version.download("yolov11")

data_yaml_path = os.path.join(dataset.location, "data.yaml")
print(f"Dataset berhasil diunduh ke: {data_yaml_path}")

# Tampilkan isi data.yaml
with open(data_yaml_path, 'r') as f:
    print("\n--- Isi data.yaml ---")
    print(f.read())

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov11 in progress : 93.0%
Version export complete for yolov11 format



Extracting Dataset Version Zip to My-First-Project-4 in yolov11:: 100%|██████████| 605/605 [00:00<00:00, 3579.89it/s]

Dataset berhasil diunduh ke: /kaggle/working/My-First-Project-4/data.yaml

--- Isi data.yaml ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['Black-faced spoonbill']

roboflow:
  workspace: blackfaced-spoonbill
  project: my-first-project-cn7qx
  version: 4
  license: CC BY 4.0
  url: https://universe.roboflow.com/blackfaced-spoonbill/my-first-project-cn7qx/dataset/4


In [3]:
# [Cell 3] Training YOLO11n-seg menggunakan Dual T4 (DDP device=[0, 1])
model_nano = YOLO("yolo11n-seg.pt")

results_nano = model_nano.train(
    data=data_yaml_path,
    epochs=100,
    patience=15,
    imgsz=640,
    batch=32,            # 32 gambar per batch (dibagi 16 per GPU T4)
    device=[0, 1],       # Dual GPU Distributed Data Parallel (DDP)
    workers=4,
    optimizer='auto',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    close_mosaic=10,
    project="Spoonbill_V3_Training",
    name="YOLO11n_DualT4",
    save=True,
    plots=True,
    verbose=True
)
print("✅ Training YOLO11n-seg Selesai!")

Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/My-First-Project-4/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      2/100      3.03G     0.9835      1.785      2.182       1.21          0          2        640: 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.7it/s 0.3s
                   all          7         24     0.0977       0.37      0.206      0.141    0.00848      0.458     0.0423     0.0127

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      3/100      3.03G     0.9958      1.428      1.623      1.196          0          3        640: 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.6it/s 0.2s
                   all          7         24    0.0

In [4]:
# [Cell 4] Training YOLO11l-seg menggunakan Dual T4
model_large = YOLO("yolo11l-seg.pt")

results_large = model_large.train(
    data=data_yaml_path,
    epochs=100,
    patience=15,
    imgsz=640,
    batch=16,            # 16 gambar per batch (dibagi 8 per GPU T4)
    device=[0, 1],       # Dual GPU DDP
    workers=4,
    optimizer='auto',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    close_mosaic=10,
    project="Spoonbill_V3_Training",
    name="YOLO11l_DualT4",
    save=True,
    plots=True,
    verbose=True
)
print("✅ Training YOLO11l-seg Selesai!")

Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/My-First-Project-4/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      2/100      6.15G      1.701      2.611      2.655      1.951          0          6        640: 100% ━━━━━━━━━━━━ 19/19 2.3it/s 8.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 7.2it/s 0.1s
                   all          7         24          0          0          0          0          0          0          0          0

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      3/100      6.19G      1.736       2.39      2.136      2.004          0          2        640: 100% ━━━━━━━━━━━━ 19/19 2.3it/s 8.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 7.2it/s 0.1s
                   all          7         24       

In [5]:
# [Cell 5] Ekspor Model Terbaik ke Format ONNX Full-Precision (Non-Quantized)
output_dir = "/kaggle/working/Exported_Models_V3"
os.makedirs(output_dir, exist_ok=True)

# 1. Ekspor Nano
best_nano_pt = "Spoonbill_V3_Training/YOLO11n_DualT4/weights/best.pt"
if os.path.exists(best_nano_pt):
    m_nano = YOLO(best_nano_pt)
    m_nano.export(format="onnx", imgsz=640, dynamic=False, simplify=True, half=False)
    shutil.copy2(best_nano_pt.replace(".pt", ".onnx"), os.path.join(output_dir, "best_spoonbill_11n.onnx"))
    shutil.copy2(best_nano_pt, os.path.join(output_dir, "best_spoonbill_11n.pt"))
    print("✅ Model Nano berhasil diekspor ke ONNX!")

# 2. Ekspor Large
best_large_pt = "Spoonbill_V3_Training/YOLO11l_DualT4/weights/best.pt"
if os.path.exists(best_large_pt):
    m_large = YOLO(best_large_pt)
    m_large.export(format="onnx", imgsz=640, dynamic=False, simplify=True, half=False)
    shutil.copy2(best_large_pt.replace(".pt", ".onnx"), os.path.join(output_dir, "best_spoonbill_11l.onnx"))
    shutil.copy2(best_large_pt, os.path.join(output_dir, "best_spoonbill_11l.pt"))
    print("✅ Model Large berhasil diekspor ke ONNX!")

# 3. Salin kurva evaluasi & confusion matrix
metrics_dir = os.path.join(output_dir, "Evaluation_Plots")
os.makedirs(metrics_dir, exist_ok=True)
for model_name in ["YOLO11n_DualT4", "YOLO11l_DualT4"]:
    src = f"Spoonbill_V3_Training/{model_name}"
    dst = os.path.join(metrics_dir, model_name)
    if os.path.exists(src):
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns("*.pt"))

# 4. Buat File ZIP Siap Download dari Kaggle Output
shutil.make_archive("/kaggle/working/Spoonbill_Models_V3_KaggleOutput", 'zip', output_dir)
print("\n🎉 SELURUH PROSES SELESAI! File ZIP siap diunduh dari tab Output Kaggle:")
print("👉 /kaggle/working/Spoonbill_Models_V3_KaggleOutput.zip")


🎉 SELURUH PROSES SELESAI! File ZIP siap diunduh dari tab Output Kaggle:
👉 /kaggle/working/Spoonbill_Models_V3_KaggleOutput.zip
